In [1]:
# ===== RESTAURANT CHAIN DATA GENERATION =====

import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
random.seed(42)

# ===== 1. BRANCHES TABLE =====
branches = pd.DataFrame({
    'branch_id':     [f'BR{str(i).zfill(3)}' for i in range(1, 16)],
    'branch_name':   [f'Branch {i}' for i in range(1, 16)],
    'city':          ['Lahore']*5 + ['Karachi']*4 + ['Islamabad']*3 + ['Faisalabad']*2 + ['Multan']*1,
    'area':          ['Gulberg','DHA','Johar Town','Model Town','Bahria Town',
                      'Clifton','Defence','Gulshan','North Nazimabad',
                      'F-7','F-10','G-11',
                      'Canal Road','Peoples Colony',
                      'Gulgasht'],
    'opening_year':  [2018, 2019, 2019, 2020, 2021,
                      2018, 2019, 2020, 2021,
                      2019, 2020, 2022,
                      2020, 2021,
                      2022],
    'seating_capacity': np.random.choice([40, 60, 80, 100, 120], 15),
    'has_delivery':  [True]*12 + [False]*3,
    'manager':       [f'Manager_{i}' for i in range(1, 16)]
})

print(f"Branches: {branches.shape}")
print(branches.head())

Branches: (15, 8)
  branch_id branch_name    city         area  opening_year  seating_capacity  \
0     BR001    Branch 1  Lahore      Gulberg          2018               100   
1     BR002    Branch 2  Lahore          DHA          2019               120   
2     BR003    Branch 3  Lahore   Johar Town          2019                80   
3     BR004    Branch 4  Lahore   Model Town          2020               120   
4     BR005    Branch 5  Lahore  Bahria Town          2021               120   

   has_delivery    manager  
0          True  Manager_1  
1          True  Manager_2  
2          True  Manager_3  
3          True  Manager_4  
4          True  Manager_5  


In [2]:
# ===== 2. MENU ITEMS TABLE =====
menu_data = {
    'item_id': [f'ITM{str(i).zfill(3)}' for i in range(1, 41)],
    'item_name': [
        # Burgers
        'Classic Burger', 'Zinger Burger', 'BBQ Burger', 'Double Patty Burger', 'Chicken Burger',
        # Pizzas
        'Margherita Pizza', 'BBQ Chicken Pizza', 'Pepperoni Pizza', 'Veggie Pizza', 'Beef Pizza',
        # Rolls
        'Chicken Roll', 'Beef Roll', 'Zinger Roll', 'Club Roll', 'Veggie Roll',
        # Deals
        'Family Deal 1', 'Family Deal 2', 'Couple Deal', 'Student Deal', 'Midnight Deal',
        # Sides
        'French Fries', 'Onion Rings', 'Coleslaw', 'Garlic Bread', 'Nuggets',
        # Drinks
        'Coca Cola', 'Sprite', 'Fanta', 'Water Bottle', 'Fresh Juice',
        # Desserts
        'Chocolate Brownie', 'Ice Cream', 'Cheesecake', 'Waffle', 'Fruit Trifle',
        # Breakfast
        'Breakfast Platter', 'Omelette', 'Pancakes', 'French Toast', 'Paratha Roll'
    ],
    'category': (
        ['Burger']*5 + ['Pizza']*5 + ['Roll']*5 +
        ['Deal']*5 + ['Sides']*5 + ['Drinks']*5 +
        ['Dessert']*5 + ['Breakfast']*5
    ),
    'price': [
        650, 750, 800, 950, 700,           # Burgers
        1100, 1300, 1250, 1000, 1200,      # Pizzas
        450, 500, 550, 480, 400,           # Rolls
        2500, 3000, 1800, 1200, 1500,      # Deals
        350, 380, 250, 300, 420,           # Sides
        150, 150, 150, 80, 280,            # Drinks
        450, 350, 500, 420, 380,           # Desserts
        800, 500, 600, 450, 350            # Breakfast
    ],
    'cost': [
        300, 350, 380, 450, 320,
        500, 600, 580, 460, 550,
        200, 230, 250, 220, 180,
        1200, 1450, 870, 580, 720,
        150, 160, 100, 130, 180,
        60, 60, 60, 30, 120,
        180, 140, 200, 170, 160,
        350, 220, 270, 200, 150
    ]
}

menu = pd.DataFrame(menu_data)
menu['profit_margin'] = ((menu['price'] - menu['cost']) / menu['price'] * 100).round(2)

print(f"Menu Items: {menu.shape}")
print(menu.groupby('category')['price'].mean().round(0))

Menu Items: (40, 6)
category
Breakfast     540.0
Burger        770.0
Deal         2000.0
Dessert       420.0
Drinks        162.0
Pizza        1170.0
Roll          476.0
Sides         340.0
Name: price, dtype: float64


In [5]:
# ===== 3. ORDERS TABLE — 100,000 orders =====
print("Generating 100,000 orders... please wait")

start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 12, 31)
date_range = (end_date - start_date).days

n_orders = 100000

dates     = [start_date + timedelta(days=random.randint(0, date_range)) for _ in range(n_orders)]
date_objs = pd.to_datetime(dates)

# Auto normalize karo taake sum exactly 1.0 ho
def normalize(weights):
    w = np.array(weights, dtype=float)
    return w / w.sum()

orders = pd.DataFrame({
    'order_id':     [f'ORD{str(i).zfill(6)}' for i in range(1, n_orders+1)],
    'branch_id':    np.random.choice(branches['branch_id'], n_orders,
                        p=normalize([12,11,10,9,8,10,9,7,6,7,5,3,2,1,1])),
    'order_date':   date_objs,
    'order_month':  date_objs.month,
    'order_year':   date_objs.year,
    'day_of_week':  date_objs.day_name(),
    'hour':         np.random.choice(
                        [12,13,14,19,20,21,22,10,11,15,16,17,18],
                        n_orders,
                        p=normalize([10,9,8,12,13,11,9,6,6,5,4,4,3])),
    'customer_type':  np.random.choice(['Dine-in','Takeaway','Delivery'],
                          n_orders, p=normalize([40,25,35])),
    'payment_method': np.random.choice(['Cash','Card','JazzCash','EasyPaisa'],
                          n_orders, p=normalize([45,30,15,10])),
    'discount_applied': np.random.choice([True,False],
                            n_orders, p=normalize([25,75])),
    'feedback_score': np.random.choice([1,2,3,4,5],
                          n_orders, p=normalize([3,7,15,40,35]))
})

print(f"Orders generated: {orders.shape}")
print(orders['customer_type'].value_counts())

Generating 100,000 orders... please wait
Orders generated: (100000, 11)
customer_type
Dine-in     39911
Delivery    35114
Takeaway    24975
Name: count, dtype: int64


In [6]:
# ===== 4. ORDER DETAILS TABLE =====
print("Generating order details... please wait")

order_details_list = []

for order_id in orders['order_id']:
    # Har order mein 1-5 items
    n_items = np.random.choice([1,2,3,4,5], p=[0.30,0.35,0.20,0.10,0.05])
    selected_items = menu.sample(n_items)

    for _, item in selected_items.iterrows():
        qty = np.random.choice([1,2,3], p=[0.70,0.25,0.05])
        discount_pct = np.random.choice([0, 10, 15, 20], p=[0.70,0.15,0.10,0.05])
        actual_price = item['price'] * qty * (1 - discount_pct/100)

        order_details_list.append({
            'order_id':     order_id,
            'item_id':      item['item_id'],
            'item_name':    item['item_name'],
            'category':     item['category'],
            'quantity':      qty,
            'unit_price':   item['price'],
            'discount_pct': discount_pct,
            'actual_price': round(actual_price, 2),
            'cost':         item['cost'] * qty
        })

order_details = pd.DataFrame(order_details_list)
order_details['profit'] = order_details['actual_price'] - order_details['cost']

print(f"Order details generated: {order_details.shape}")
print(f"Total revenue: Rs. {order_details['actual_price'].sum():,.0f}")

Generating order details... please wait
Order details generated: (225263, 10)
Total revenue: Rs. 214,209,893


In [ ]:
# ===== 5. SAVE ALL TABLES =====
branches.to_csv('../data/branches.csv', index=False)
menu.to_csv('../data/menu_items.csv', index=False)
orders.to_csv('../data/orders.csv', index=False)
order_details.to_csv('../data/order_details.csv', index=False)

# Master table for Power BI
master = orders.merge(order_details, on='order_id')
master = master.merge(branches[['branch_id','city','area']], on='branch_id')
master.to_csv('../data/master_data.csv', index=False)

print("="*45)
print("ALL FILES SAVED SUCCESSFULLY!")
print("="*45)
print(f"Branches      : {branches.shape[0]} rows")
print(f"Menu Items    : {menu.shape[0]} rows")
print(f"Orders        : {orders.shape[0]} rows")
print(f"Order Details : {order_details.shape[0]} rows")
print(f"Master Table  : {master.shape[0]} rows")
print(f"\nTotal Revenue : Rs. {master['actual_price'].sum():,.0f}")
print(f"Total Profit  : Rs. {master['profit'].sum():,.0f}")

ALL FILES SAVED SUCCESSFULLY!
Branches      : 15 rows
Menu Items    : 40 rows
Orders        : 100000 rows
Order Details : 225263 rows
Master Table  : 225263 rows

Total Revenue : Rs. 214,209,893
Total Profit  : Rs. 111,874,843
